# A/B тест: аналіз депозитної активності

Цей ноутбук проходить весь аналіз крок за кроком.
Запускай клітинки по черзі зверху вниз (Shift+Enter або кнопка Run).

**Файли що потрібні поруч з ноутбуком:**
- `users.xlsx` — user_id, segment, group
- `inout.xlsx` — user_id, type, amount_eur
- `bonuses.xlsx` — bonus_id (список 3 тестових бонусів)
- `activations.xlsx` — user_id, bonus_id, bonus_type, amount_eur

---
## Клітинка 1. Імпорт бібліотек

Завантажуємо все необхідне. Якщо якась бібліотека не встановлена,
запусти в терміналі: `pip install pandas scipy openpyxl`

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Налаштування відображення таблиць
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

print('Бібліотеки завантажено')

---
## Клітинка 2. Завантаження файлів

Змінни назви файлів якщо вони у тебе інші.
Також зміни назви колонок якщо вони відрізняються від вказаних нижче.

In [ ]:
# Завантажуємо файли
users       = pd.read_excel('users.xlsx')
inout       = pd.read_excel('inout.xlsx')
bonuses     = pd.read_excel('bonuses.xlsx')
activations = pd.read_excel('activations.xlsx')

# --- Назви колонок ---
# Якщо у тебе колонки називаються інакше, заміни значення праворуч
# Наприклад якщо замість 'user_id' у тебе 'UserID' — пиши 'UserID'

COL_USER_ID    = 'user_id'
COL_SEGMENT    = 'segment'
COL_GROUP      = 'group'
COL_TX_TYPE    = 'type'
COL_TX_AMOUNT  = 'amount_eur'
COL_BONUS_ID   = 'bonus_id'
COL_BONUS_TYPE = 'bonus_type'
COL_BONUS_AMT  = 'amount_eur'

# Перевірка: виводимо перші рядки кожного файлу
print('=== users ===')
print(users.head(3))
print('\n=== inout ===')
print(inout.head(3))
print('\n=== bonuses ===')
print(bonuses.head(3))
print('\n=== activations ===')
print(activations.head(3))

---
## Клітинка 3. Фільтрація даних

З файлу inout залишаємо тільки юзерів що є в тесті.
З файлу activations залишаємо тільки активації тестових бонусів.

Юзери що не в тесті нас не цікавлять — вони не отримували листа.

In [ ]:
# Список id юзерів що беруть участь в тесті
test_users = set(users[COL_USER_ID])

# Список id тестових бонусів
test_bonuses = set(bonuses[COL_BONUS_ID])

# Фільтруємо транзакції — тільки тестові юзери
tx = inout[inout[COL_USER_ID].isin(test_users)].copy()

# Фільтруємо активації — тільки тестові бонуси
act = activations[activations[COL_BONUS_ID].isin(test_bonuses)].copy()

print(f'Юзерів в тесті:                  {len(test_users)}')
print(f'Транзакцій після фільтрації:     {len(tx)}')
print(f'Активацій тестових бонусів:      {len(act)}')
print()

# Перевірка: скільки унікальних типів транзакцій
print('Типи транзакцій:', tx[COL_TX_TYPE].unique())

---
## Клітинка 4. Contamination check

Перевіряємо чи активував хтось з групи A тестові бонуси.
Такого бути не повинно — якщо є, це помилка в розбивці груп.

In [ ]:
# Юзери групи A
group_a_users = set(users[users[COL_GROUP] == 'A'][COL_USER_ID])

# Юзери що активували тестові бонуси
users_with_bonus = set(act[COL_USER_ID])

# Перетин — юзери A що активували тестові бонуси
contaminated = group_a_users & users_with_bonus

print(f'Юзерів групи A що активували тестові бонуси: {len(contaminated)}')

if len(contaminated) > 0:
    print('Є contamination. Ці юзери будуть виключені з аналізу.')
    # Виключаємо їх з users
    users = users[~users[COL_USER_ID].isin(contaminated)].copy()
    tx    = tx[~tx[COL_USER_ID].isin(contaminated)].copy()
    print(f'Залишилось юзерів: {len(users)}')
else:
    print('Contamination немає, все чисто.')

---
## Клітинка 5. Агрегація транзакцій

Один юзер може мати багато рядків в inout.
Зводимо їх до одного рядка на юзера:
окремо рахуємо суму всіх in і суму всіх out.

In [ ]:
# Важливо: заміни 'in' і 'out' на реальні значення з твого файлу
# Їх видно у виводі клітинки 3 де друкуються типи транзакцій
TYPE_IN  = 'in'
TYPE_OUT = 'out'

# Агрегація депозитів
tx_in = (
    tx[tx[COL_TX_TYPE] == TYPE_IN]
    .groupby(COL_USER_ID)[COL_TX_AMOUNT]
    .agg(total_in='sum', deposit_count='count')
    .reset_index()
)

# Агрегація виплат
tx_out = (
    tx[tx[COL_TX_TYPE] == TYPE_OUT]
    .groupby(COL_USER_ID)[COL_TX_AMOUNT]
    .agg(total_out='sum')
    .reset_index()
)

print(f'Юзерів з хоча б одним депозитом: {len(tx_in)}')
print(f'Юзерів з хоча б одною виплатою:  {len(tx_out)}')

---
## Клітинка 6. Агрегація бонусів

Зводимо активації до одного рядка на юзера.
Рахуємо загальну суму бонусів і кількість активацій по кожному типу.

In [ ]:
# Загальна сума і кількість активацій на юзера
bonus_agg = (
    act
    .groupby(COL_USER_ID)[COL_BONUS_AMT]
    .agg(bonus_cost='sum', bonus_count='count')
    .reset_index()
)
bonus_agg['activated_flag'] = 1  # якщо рядок є — юзер активував хоча б один бонус

# Кількість активацій по кожному типу бонусу окремо
bonus_by_type = (
    act
    .pivot_table(
        index=COL_USER_ID,
        columns=COL_BONUS_TYPE,
        values=COL_BONUS_ID,
        aggfunc='count',
        fill_value=0
    )
    .reset_index()
)

print('Типи бонусів:', act[COL_BONUS_TYPE].unique())
print(f'Юзерів що активували хоча б один тестовий бонус: {len(bonus_agg)}')

---
## Клітинка 7. Збірка майстер-таблиці

Джойнимо всі агрегати до базового списку юзерів.
Використовуємо left join щоб зберегти всіх юзерів,
навіть тих хто не мав транзакцій або не активував бонуси.
Для них проставляємо 0.

In [ ]:
master = users.copy()

master = master.merge(tx_in,       on=COL_USER_ID, how='left')
master = master.merge(tx_out,      on=COL_USER_ID, how='left')
master = master.merge(bonus_agg,   on=COL_USER_ID, how='left')
master = master.merge(bonus_by_type, on=COL_USER_ID, how='left')

# Замінюємо NaN на 0 для числових колонок
# NaN виникає коли юзер не мав транзакцій або не активував бонус
fill_zero = ['total_in', 'deposit_count', 'total_out',
             'bonus_cost', 'bonus_count', 'activated_flag']
master[fill_zero] = master[fill_zero].fillna(0)

# Типи бонусів теж заповнюємо нулями
bonus_type_cols = [c for c in master.columns
                   if c not in list(users.columns) + fill_zero +
                   ['total_out', 'bonus_cost', 'bonus_count']]
master[bonus_type_cols] = master[bonus_type_cols].fillna(0)

# Розраховуємо похідні метрики
master['ngr'] = master['total_in'] - master['total_out'] - master['bonus_cost']

# out_rate: для юзерів без депозиту залишаємо NaN (ділення на 0 не має сенсу)
master['out_rate'] = np.where(
    master['total_in'] > 0,
    master['total_out'] / master['total_in'] * 100,
    np.nan
)

print(f'Рядків в майстер-таблиці: {len(master)}')
print(f'Колонки: {list(master.columns)}')
print()
master.head()

---
## Клітинка 8. Перевірка якості даних

Перед аналізом переконуємось що все зібрано правильно.

In [ ]:
print('--- Розмір груп ---')
print(master[COL_GROUP].value_counts())

print('\n--- Розподіл сегментів по групах (частки) ---')
seg_dist = master.groupby([COL_GROUP, COL_SEGMENT]).size().unstack(fill_value=0)
print(seg_dist.div(seg_dist.sum(axis=1), axis=0).round(3))

print('\n--- Юзери без жодного депозиту ---')
no_deposit = (master['total_in'] == 0).sum()
print(f'Всього: {no_deposit} ({no_deposit/len(master)*100:.1f}%)')

print('\n--- Активація бонусів по групах ---')
print(master.groupby(COL_GROUP)['activated_flag'].sum())

print('\n--- Базова статистика по ключових метриках ---')
master.groupby(COL_GROUP)[['total_in', 'total_out', 'ngr', 'deposit_count']].describe().round(2)

---
## Клітинка 9. Функція для статистичних тестів

Тут описані два типи тестів:

- **Mann-Whitney** — для числових метрик (суми, кількості).
  Не потребує нормального розподілу, підходить для даних казино
  де є багато нулів і поодинокі великі суми.

- **Chi-square** — для відсоткових метрик (конверсія в депозит,
  частка юзерів що активували бонус).

Ця клітинка тільки визначає функцію — вона нічого не рахує сама по собі.
Результати побачиш в наступних клітинках.

In [ ]:
def run_analysis(df, label='Overall'):
    """
    Приймає датафрейм (або підмножину по сегменту),
    рахує метрики і статтести для груп A vs B.
    Повертає датафрейм з результатами.
    """
    a = df[df[COL_GROUP] == 'A']
    b = df[df[COL_GROUP] == 'B']

    results = []

    # Числові метрики — Mann-Whitney
    numeric_metrics = {
        'total_in':      'IN (сума депозитів, EUR)',
        'total_out':     'OUT (сума виплат, EUR)',
        'ngr':           'NGR (EUR)',
        'out_rate':      'Out rate (%)',
        'deposit_count': 'Кількість депозитів',
    }

    for col, name in numeric_metrics.items():
        a_vals = a[col].dropna()
        b_vals = b[col].dropna()

        # Пропускаємо якщо даних замало
        if len(a_vals) < 5 or len(b_vals) < 5:
            continue

        stat, p = stats.mannwhitneyu(a_vals, b_vals, alternative='two-sided')

        mean_a = a_vals.mean()
        mean_b = b_vals.mean()
        lift   = (mean_b - mean_a) / mean_a * 100 if mean_a != 0 else None

        results.append({
            'Метрика':       name,
            'Середнє A':     round(mean_a, 2),
            'Середнє B':     round(mean_b, 2),
            'Lift %':        round(lift, 1) if lift is not None else None,
            'p-value':       round(p, 4),
            'Значимо':       'так' if p < 0.05 else 'ні',
            'Тест':          'Mann-Whitney',
        })

    # Бінарні метрики — Chi-square
    binary_metrics = {
        'deposited':      ('Конверсія в депозит (%)', lambda d: (d['total_in'] > 0).astype(int)),
        'activated_flag': ('Активація бонусу (%)',    lambda d: d['activated_flag'].astype(int)),
    }

    for col, (name, getter) in binary_metrics.items():
        a_vals = getter(a)
        b_vals = getter(b)

        # Таблиця спряженості для chi-square
        table = [
            [a_vals.sum(), len(a_vals) - a_vals.sum()],
            [b_vals.sum(), len(b_vals) - b_vals.sum()],
        ]

        chi2, p, _, _ = stats.chi2_contingency(table)

        rate_a = a_vals.mean() * 100
        rate_b = b_vals.mean() * 100
        lift   = (rate_b - rate_a) / rate_a * 100 if rate_a != 0 else None

        results.append({
            'Метрика':   name,
            'Середнє A': f'{round(rate_a, 1)}%',
            'Середнє B': f'{round(rate_b, 1)}%',
            'Lift %':    round(lift, 1) if lift is not None else None,
            'p-value':   round(p, 4),
            'Значимо':   'так' if p < 0.05 else 'ні',
            'Тест':      'Chi-square',
        })

    res = pd.DataFrame(results)

    # Попередження про малу вибірку
    small_sample = min(len(a), len(b)) < 30

    print(f'\n{"-"*55}')
    print(f'{label}  |  A: {len(a)} юзерів  |  B: {len(b)} юзерів')
    if small_sample:
        print('  Мала вибірка — результати орієнтовні')
    print(f'{"-"*55}')
    print(res.to_string(index=False))

    return res

print('Функцію визначено, можна переходити до наступної клітинки')

---
## Клітинка 10. Загальний аналіз A vs B

In [ ]:
overall = run_analysis(master, label='OVERALL')

---
## Клітинка 11. Аналіз по сегментах

Той самий аналіз але окремо для кожного з 5 сегментів.
Ефект може суттєво відрізнятись між сегментами.

In [ ]:
segment_results = {}

for seg in sorted(master[COL_SEGMENT].unique()):
    seg_df = master[master[COL_SEGMENT] == seg]
    segment_results[seg] = run_analysis(seg_df, label=f'Сегмент: {seg}')

---
## Клітинка 12. Зведена таблиця лифтів по сегментах

Компактний вид: один рядок = один сегмент,
колонки = lift по кожній метриці.
Зручно щоб порівняти сегменти між собою одним поглядом.

In [ ]:
rows = []
for seg, res in segment_results.items():
    row = {'segment': seg}
    for _, r in res.iterrows():
        # Формат: значення lift + позначка значимості
        sig = ' *' if r['Значимо'] == 'так' else ''
        row[r['Метрика']] = f"{r['Lift %']}{sig}"
    rows.append(row)

lift_summary = pd.DataFrame(rows).set_index('segment')

print('Lift B vs A по сегментах (* = статистично значимо на рівні p < 0.05)')
print()
lift_summary

---
## Клітинка 13. Аналіз активації бонусів у групі B

Дивимось тільки на групу B.
Порівнюємо юзерів що активували тестовий бонус
з тими хто не активував — по метриках IN і NGR.

Увага: це описова статистика, не причинно-наслідковий зв'язок.
Активатори могли бути активнішими ще до тесту.

In [ ]:
group_b = master[master[COL_GROUP] == 'B'].copy()

print('--- Група B: активатори vs неактиватори ---')
print()
print(group_b.groupby('activated_flag')[['total_in', 'ngr', 'deposit_count']]
      .agg(['mean', 'median', 'count'])
      .round(2))

# Якщо є колонки типів бонусів — показуємо activation rate по кожному
bonus_types = act[COL_BONUS_TYPE].unique()
print()
print('--- Activation rate по типах бонусів (група B) ---')
for bt in bonus_types:
    if bt in group_b.columns:
        activated = (group_b[bt] > 0).sum()
        rate = activated / len(group_b) * 100
        print(f'  {bt}: {activated} юзерів ({rate:.1f}%)')

---
## Клітинка 14. Збереження результатів в Excel

Записуємо все в один файл з кількома листами:
- Overall — загальний аналіз
- Lift по сегментах — зведена таблиця
- Окремий лист для кожного сегменту
- Master data — повна таблиця на юзера

In [ ]:
output_file = 'ab_test_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    overall.to_excel(writer, sheet_name='Overall', index=False)

    lift_summary.to_excel(writer, sheet_name='Lift по сегментах')

    for seg, res in segment_results.items():
        # Excel обмежує назву листа 31 символом
        sheet_name = str(seg)[:31]
        res.to_excel(writer, sheet_name=sheet_name, index=False)

    master.to_excel(writer, sheet_name='Master data', index=False)

print(f'Збережено в {output_file}')